In [16]:
# Libraries
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [17]:
DATA_PATH = "/content/final-data-2022 copy.csv"      # change this
TARGET = "crime_count"
LSOA_COL = "LSOA code"

In [18]:
# Feature engineering + loading data

# calendar columns
POSSIBLE_EXTRA_FEATURES = [
    "month_sin", "month_cos",
]

# Load and prep
df = pd.read_csv(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"], errors="raise")
df.sort_values([LSOA_COL, "Date"], inplace=True)

# Build lag features
def add_lags(g):
    g = g.copy()
    g["crime_count_lag1"]       = g[TARGET].shift(1)
    g["crime_count_rolling3"]   = g[TARGET].rolling(3).mean().shift(1)
    g["crime_count_rolling12"]  = g[TARGET].rolling(12).mean().shift(1)
    return g

df = df.groupby(LSOA_COL, group_keys=False).apply(add_lags)

# drop na rows (not existing i believe)
df.dropna(subset=["crime_count_lag1"], inplace=True)

# Train-test split
test_mask   = df["Date"].dt.to_period("M") == pd.Period("2025-04")
train_mask  = df["Date"].dt.to_period("M") <  pd.Period("2025-04")

df_train = df.loc[train_mask].copy()
df_test  = df.loc[test_mask].copy()

# Feature set
feature_cols = (
    ["crime_count_lag1", "crime_count_rolling3", "crime_count_rolling12"]
    + [c for c in POSSIBLE_EXTRA_FEATURES if c in df.columns]
)

print("Features being used:", feature_cols)

<ipython-input-18-939494980>:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(LSOA_COL, group_keys=False).apply(add_lags)


Features being used: ['crime_count_lag1', 'crime_count_rolling3', 'crime_count_rolling12', 'month_sin', 'month_cos']


In [19]:
# Training linear regressions
y_true_all, y_pred_all, lsoa_ids = [], [], []

model_tpl = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
        ("ols",     LinearRegression())
    ]
)

for lsoa, grp_train in df_train.groupby(LSOA_COL, as_index=False):
    grp_test = df_test[df_test[LSOA_COL] == lsoa]
    if grp_test.empty:
        continue            # no Apr-2025 row for this LSOA

    X_train, y_train = grp_train[feature_cols], grp_train[TARGET]
    X_test,  y_test  = grp_test[feature_cols],  grp_test[TARGET]

    if len(y_train) < 2:
        continue            # not enough history to fit

    mdl = model_tpl.fit(X_train, y_train)
    y_pred = mdl.predict(X_test)

    y_true_all.extend(y_test)
    y_pred_all.extend(y_pred)
    lsoa_ids.extend([lsoa] * len(y_pred))

In [21]:
# ── Evaluation ────────────────────────────────────────────────────────────────────
mae  = mean_absolute_error(y_true_all, y_pred_all)
rmse = mean_squared_error(y_true_all, y_pred_all)

print("\nHold-out metrics for April-2025")
print(f"  MAE : {mae:0.4f}")
print(f"  RMSE: {rmse:0.4f}\n")


Hold-out metrics for April-2025
  MAE : 0.8003
  RMSE: 1.2165

